<a href="https://colab.research.google.com/github/Clanboy777/THE-OP-BANK-OF-6-7/blob/main/Real_vcb_worldplayer67.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

YOUTUBE_API_KEY = "" # <--- **CRITICAL**: Replace this entire string with your personal, valid YouTube API key

if YOUTUBE_API_KEY == "REPLACE_THIS_WITH_YOUR_ACTUAL_YOUTUBE_API_KEY":
    print("⚠️ Add your YouTube API key in this cell.")
else:
    print("✅ YouTube API key added.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print("🤖 Loading Alpha AI...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

model.eval()

print("✅ Alpha AI loaded!")

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Hello Alpha AI! Introduce yourself."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("🤖 Alpha AI:")
print(answer)

In [ ]:
# CELL 4 — Alpha AI Flask + World Mode + World YouTube Sync

from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from flask_socketio import SocketIO, emit
import requests

app = Flask(__name__)
app.config["SECRET_KEY"] = "alpha-ai-secret"
CORS(app)
socketio = SocketIO(app, cors_allowed_origins="*", async_mode="threading")

solo_chats = {}
world_messages = []
connected_users = {}

# Shared World YouTube state
world_player_state = {
    "current": None,
    "queue": []
}

SYSTEM_PROMPT = """
You are Alpha AI, a helpful and friendly AI assistant.

Give clear, useful and natural answers.

You are called Alpha AI.

If the user asks:
who made you
who created you
who built you
who programmed you
who is your creator
who is your maker

answer exactly:

I was made by Akshat. 🤖
"""

creator_questions = [
    "who made you",
    "who created you",
    "who built you",
    "who programmed you",
    "who is your creator",
    "who is your maker"
]


def generate_response(history, user_message):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history)
    messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer or "I'm ready. 🤖"


@app.route("/")
def home():
    return send_file("index.html")


@app.route("/chat", methods=["POST"])
def chat():
    data = request.get_json(silent=True) or {}
    username = (data.get("username") or "Guest").strip() or "Guest"
    message = (data.get("message") or "").strip()

    if not message:
        return jsonify({"reply": "Please enter a message."})

    history = solo_chats.setdefault(username, [])
    lower_message = message.lower().strip()

    if any(q in lower_message for q in creator_questions):
        answer = "I was made by Akshat. 🤖"
    else:
        answer = generate_response(history, message)

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})

    if len(history) > 40:
        del history[:-40]

    return jsonify({"reply": answer})


@app.route("/reset", methods=["POST"])
def reset():
    data = request.get_json(silent=True) or {}
    username = (data.get("username") or "Guest").strip() or "Guest"
    solo_chats.pop(username, None)
    return jsonify({"ok": True})


@app.route("/music_search", methods=["POST"])
def music_search():
    data = request.get_json(silent=True) or {}
    query = (data.get("query") or "").strip()

    if not query:
        return jsonify({"error": "Please enter a song name."}), 400

    key = globals().get("YOUTUBE_API_KEY", "")
    if not key or key == "PASTE_YOUR_YOUTUBE_API_KEY_HERE":
        return jsonify({"error": "YouTube API key is not configured in Cell 1."}), 500

    try:
        response = requests.get(
            "https://www.googleapis.com/youtube/v3/search",
            params={
                "part": "snippet",
                "q": query,
                "type": "video",
                "maxResults": 1,
                "key": key
            },
            timeout=7
        )
        data = response.json()

        if response.status_code != 200:
            error_message = data.get("error", {}).get("message", "YouTube search failed.")
            return jsonify({"error": error_message}), response.status_code

        items = data.get("items", [])
        if not items:
            return jsonify({"error": "No YouTube video found."}), 404

        item = items[0]
        return jsonify({
            "title": item["snippet"]["title"],
            "videoId": item["id"]["videoId"]
        })

    except requests.RequestException as e:
        print("YOUTUBE SEARCH ERROR:", e)
        return jsonify({"error": "YouTube search timed out or failed."}), 504
    except Exception as e:
        print("YOUTUBE ERROR:", e)
        return jsonify({"error": "YouTube search failed."}), 500


# =========================
# WORLD MODE
# =========================

@socketio.on("join_world")
def join_world(data):
    username = (data.get("username") or "Guest").strip() or "Guest"
    connected_users[request.sid] = username

    emit("world_history", world_messages)
    emit("world_player_state", world_player_state)

    socketio.emit(
        "system_message",
        {"message": username + " joined the world 🌐"}
    )


@socketio.on("world_message")
def world_message(data):
    username = connected_users.get(request.sid, "Guest")
    message = (data.get("message") or "").strip()
    if not message:
        return

    world_item = {
        "username": username,
        "message": message,
        "type": "user"
    }

    world_messages.append(world_item)
    if len(world_messages) > 100:
        del world_messages[:-100]

    socketio.emit("world_message", world_item)

    try:
        recent = []
        for item in world_messages[-15:]:
            recent.append({
                "role": "user",
                "content": item["username"] + ": " + item["message"]
            })

        lower_message = message.lower().strip()
        if any(q in lower_message for q in creator_questions):
            answer = "I was made by Akshat. 🤖"
        else:
            answer = generate_response(recent, message)

        ai_item = {
            "username": "Alpha AI",
            "message": answer,
            "type": "ai"
        }

        world_messages.append(ai_item)
        socketio.emit("world_message", ai_item)

    except Exception as e:
        print("WORLD AI ERROR:", e)


# =========================
# WORLD YOUTUBE SYNC
# =========================

@socketio.on("world_play_request")
def world_play_request(data):
    video = data.get("video") or {}
    if not video.get("videoId"):
        return

    world_player_state["current"] = {
        "title": video.get("title", "YouTube video"),
        "videoId": video["videoId"]
    }

    # A direct PLAY starts immediately on every connected device.
    socketio.emit("world_play", world_player_state["current"])

    username = connected_users.get(request.sid, "Guest")
    socketio.emit("system_message", {
        "message": username + " is playing: " + world_player_state["current"]["title"] + " 🎵"
    })


@socketio.on("world_queue_add_request")
def world_queue_add_request(data):
    video = data.get("video") or {}
    if not video.get("videoId"):
        return

    item = {
        "title": video.get("title", "YouTube video"),
        "videoId": video["videoId"]
    }

    world_player_state["queue"].append(item)

    # Send the exact same resolved video to every connected device.
    socketio.emit("world_queue_add", item)

    username = connected_users.get(request.sid, "Guest")
    socketio.emit("system_message", {
        "message": username + " queued: " + item["title"] + " 🎵"
    })


@socketio.on("world_queue_remove_request")
def world_queue_remove_request(data):
    try:
        index = int(data.get("index", -1))
    except Exception:
        return

    if 0 <= index < len(world_player_state["queue"]):
        world_player_state["queue"].pop(index)
        socketio.emit("world_queue_replace", world_player_state["queue"])


@socketio.on("disconnect")
def disconnect():
    username = connected_users.pop(request.sid, "Guest")
    socketio.emit(
        "system_message",
        {"message": username + " left the world 🌐"}
    )


print("✅ Alpha AI Flask + World YouTube Sync backend ready!")


In [ ]:
html_code = r'''
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<meta name="viewport"
content="width=device-width, initial-scale=1.0">

<title>Alpha AI</title>

<script src="https://cdn.socket.io/4.7.5/socket.io.min.js"></script>
<script src="https://www.youtube.com/iframe_api"></script>

<style>

/* =========================
   BASIC
========================= */

* {
    box-sizing: border-box;
}

html,
body {

    margin: 0;
    padding: 0;

    width: 100%;
    height: 100%;

    overflow: hidden;

    background: #020402;

    color: #00ff66;

    font-family:
        Consolas,
        "Courier New",
        monospace;
}


/* =========================
   MATRIX BACKGROUND
========================= */

#matrix {

    position: fixed;

    inset: 0;

    z-index: 0;

    opacity: 0.18;

    pointer-events: none;
}


/* =========================
   CRT
========================= */

body::after {

    content: "";

    position: fixed;

    inset: 0;

    pointer-events: none;

    z-index: 100;

    background:
        repeating-linear-gradient(
            to bottom,
            rgba(255,255,255,0.025) 0px,
            rgba(255,255,255,0.025) 1px,
            transparent 2px,
            transparent 4px
        );

}


/* =========================
   APP
========================= */

#app {

    position: relative;

    z-index: 2;

    width: 100%;
    height: 100%;

    display: flex;

    flex-direction: column;
}

/* =========================
   HACKED OVERLAY
========================= */
#hackedOverlay {
    position: fixed;
    inset: 0;
    z-index: 9; /* Above normal content, below popups */
    background-color: rgba(0, 0, 0, 0.9);
    display: none; /* Hidden by default */
    pointer-events: all; /* Blocks interaction */
    backdrop-filter: blur(5px) grayscale(100%); /* Adds a strong visual effect */
}

/* =========================
   HEADER
========================= */

header {

    height: 75px;

    display: flex;

    align-items: center;

    padding: 0 20px;

    border-bottom:
        1px solid #064d24;

    background:
        rgba(0,10,5,0.88);

    position: relative;
}


#menuButton {

    font-size: 30px;

    cursor: pointer;

    color: #00ff66;

    margin-right: 18px;

    user-select: none;
}


#title {

    font-size: 28px;

    font-weight: bold;

    color: #6500b8; /* Deep purple */

    cursor: pointer;

    user-select: none;
}


#tagline {

    margin-left: 18px;

    color: #00ff66;

    font-size: 14px;
}

/* Enhanced capability glitch */
#capability.glitch {
    animation: superGlitchedText 0.1s infinite alternate;
    font-family: 'Press Start 2P', cursive; /* A glitched-like font */
    text-shadow: 2px 0 #d60000, -2px 0 #00ffff;
    transition: color 0.1s;
}

@keyframes superGlitchedText {
    0% { transform: translate(0); text-shadow: 2px 0 #d60000, -2px 0 #00ffff; filter: hue-rotate(0deg); }
    20% { transform: translate(-3px, 3px); text-shadow: -2px 0 #d60000, 2px 0 #00ffff; filter: hue-rotate(60deg); }
    40% { transform: translate(3px, -3px); text-shadow: 4px 0 #d60000, -4px 0 #00ffff; filter: hue-rotate(120deg); }
    60% { transform: translate(-1px, 1px); text-shadow: -1px 0 #d60000, 1px 0 #00ffff; filter: hue-rotate(180deg); }
    80% { transform: translate(2px, -2px); text-shadow: -4px 0 #d60000, 4px 0 #00ffff; filter: hue-rotate(240deg); }
    100% { transform: translate(0); text-shadow: 2px 0 #d60000, -2px 0 #00ffff; filter: hue-rotate(360deg); }
}

#headerWarning {
    margin-left: auto; /* Push to the right */
    color: #ffcc00; /* Warning color */
    font-size: 12px;
    padding: 5px 10px;
    border: 1px solid #ffcc00;
    border-radius: 3px;
    display: block; /* Always visible as per request */
}

/* =========================
   SIDE MENU
========================= */

#sideMenu {

    position: fixed;

    top: 0;
    left: -300px;

    width: 300px;
    height: 100%;

    z-index: 50;

    background:
        rgba(1,8,4,0.98);

    border-right:
        1px solid #00ff66;

    transition:
        left 0.25s ease;

    padding: 25px;

    box-shadow:
        0 0 30px
        rgba(0,255,100,0.15);
}


#sideMenu.open {

    left: 0;
}

.menuTitleContainer {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 25px;
}


.menuTitle {

    font-size: 22px;

    color: #6500b8;
}

.menuCloseButton {
    background: none;
    border: none;
    color: #00ff66;
    font-size: 20px;
    cursor: pointer;
    padding: 0;
}

.menuButton {

    width: 100%;

    padding: 14px;

    margin-bottom: 12px;

    border: 1px solid #064d24;

    background: #020a05;

    color: #00ff66;

    font-family: inherit;

    cursor: pointer;

    text-align: left;
}


.menuButton:hover {

    border-color: #00ff66;

    background: #031408;

}


.menuButton.active {

    border-color: #00ff66;

    box-shadow:
        0 0 10px
        rgba(0,255,100,0.25);
}


/* =========================
   MAIN
========================= */

#main {

    flex: 1;

    min-height: 0;

    display: flex;

    flex-direction: column;
}


/* =========================
   CHAT AREA
========================= */

#chatArea {

    flex: 1;

    min-height: 0;

    overflow-y: auto;

    padding: 20px;

}


/* =========================
   MESSAGES
========================= */

.message {

    max-width: 85%;

    margin-bottom: 15px;

    padding: 12px 15px;

    border-left:
        2px solid #00ff66;

    background:
        rgba(0,20,8,0.7);

    white-space: pre-wrap;

    word-wrap: break-word;
}


.userMessage {

    margin-left: auto;

    border-left:
        0;

    border-right:
        2px solid #6500b8;

    color: #c78cff;

    text-align: right;
}


.aiMessage {

    color: #00ff66;
}


.systemMessage {

    color: #888;

    text-align: center;

    font-size: 13px;

    border: 0;

    background: transparent;
}


/* =========================
   WORLD WINDOW
========================= */

#worldWindow {

    display: none;

    flex: 1;

    min-height: 0;

    overflow-y: auto;

    padding: 20px;

    border:
        1px solid #064d24;

    margin: 12px;
}


/* =========================
   YOUTUBE PLAYER
========================= */

#youtubeWindow {

    display: none;

    margin: 10px 15px;

    padding: 12px;

    border:
        1px solid #064d24;

    background:
        rgba(0,10,5,0.9);
}


#youtubeWindow iframe {

    width: 100%;

    height: 260px;

    border: 0;
}


#queueBox {

    margin-top: 12px;

    border-top: 1px solid #064d24;

    padding-top: 10px;

}


#queueHeader {

    color: #00ff66;

    font-size: 13px;

    margin-bottom: 8px;

}


#queueInputRow {

    display: flex;

    gap: 6px;

}


#queueInput {

    flex: 1;

    min-width: 0;

    background: #010401;

    color: #00ff66;

    border: 1px solid #064d24;

    padding: 8px;

    outline: none;

    font-family: inherit;

}


#queueInput:focus {

    border-color: #00ff66;

}


#queueAddButton {

    background: #020a05;

    color: #00ff66;

    border: 1px solid #064d24;

    padding: 8px 12px;

    cursor: pointer;

    font-family: inherit;

}


#queueAddButton:hover {

    border-color: #00ff66;

}


#queueList {

    margin-top: 8px;

    max-height: 120px;

    overflow-y: auto;

    font-size: 12px;

}


.queueItem {

    display: flex;

    align-items: center;

    gap: 6px;

    padding: 6px 0;

    border-bottom: 1px dotted #064d24;

}


.queueItemTitle {

    flex: 1;

    min-width: 0;

    overflow: hidden;

    text-overflow: ellipsis;

    white-space: nowrap;

}


.queueRemoveButton {

    background: transparent;

    color: #d60000;

    border: 1px solid #064d24;

    cursor: pointer;

}


#videoTitle {

    margin-bottom: 8px;

    color: #00ff66;

    overflow: hidden;

    text-overflow: ellipsis;

    white-space: nowrap;
}


.playerControls button {

    background: #020a05;

    color: #00ff66;

    border: 1px solid #064d24;

    padding: 7px 12px;

    margin-right: 5px;

    cursor: pointer;
}


/* =========================
   INPUT
========================= */

#inputArea {

    display: flex;

    gap: 8px;

    padding: 12px;

    border-top:
        1px solid #064d24;

    background:
        rgba(0,10,5,0.9);
}


#messageInput {

    flex: 1;

    min-width: 0;

    background: #010401;

    color: #00ff66;

    border:
        1px solid #064d24;

    padding: 14px;

    outline: none;

    font-family: inherit;
}


#messageInput:focus {

    border-color: #00ff66;
}


.actionButton {

    background: #020a05;

    color: #00ff66;

    border:
        1px solid #064d24;

    padding: 0 14px;

    cursor: pointer;

    font-family: inherit;
}


.actionButton:hover {

    border-color: #00ff66;
}


/* =========================
   HACKED TEXT
========================= */

#shadowHack {

    position: fixed;

    right: 15px;

    top: 85px;

    z-index: 11; /* Above hacked overlay and other content */

    font-size: 11px;

    pointer-events: none;

    opacity: 0.7;
    display: none; /* Hidden by default */
    animation: superGlitchedText 0.1s infinite alternate; /* Use more aggressive glitch */
}


#shadowHack .shadow {

    color: #6500b8; /* Deep purple */
}


#shadowHack .hacked {

    color: #d60000; /* Red */
}

/* =========================
   CREATOR POPUP
========================= */

#creatorPopup {

    display: none;

    position: fixed;

    left: 50%;
    top: 50%;

    transform:
        translate(-50%,-50%);

    z-index: 80;

    padding: 25px 35px;

    border:
        1px solid #6500b8;

    background:
        rgba(2,0,8,0.97);

    color: #6500b8;

    font-size: 20px;

    box-shadow:
        0 0 30px
        rgba(101,0,184,0.5);
}

/* =========================
   HACKED WARNING POPUPS
========================= */
.hackedWarningPopup {
    position: fixed;
    background: rgba(100, 0, 0, 0.95); /* Deep red background */
    border: 2px solid #ff0000; /* Bright red border */
    color: #ffcccc; /* Light red text */
    padding: 15px 25px;
    font-family: 'Press Start 2P', cursive; /* Glitched font */
    font-size: 16px;
    text-align: center;
    box-shadow: 0 0 20px rgba(255, 0, 0, 0.7);
    animation: glitchEffectPopup 0.2s infinite alternate, fadeInOut 1s ease-in-out;
    z-index: 100; /* Topmost */
    pointer-events: none; /* Don't block clicks */
    opacity: 0;
    display: block;
}

@keyframes glitchEffectPopup {
    0% { transform: translate(0); text-shadow: 1px 0 #ff00ff, -1px 0 #00ffff; }
    25% { transform: translate(-1px, 1px); text-shadow: -1px 0 #ff00ff, 1px 0 #00ffff; }
    50% { transform: translate(1px, -1px); text-shadow: 2px 0 #ff00ff, -2px 0 #00ffff; }
    75% { transform: translate(-1px, 0px); text-shadow: -2px 0 #ff00ff, 2px 0 #00ffff; }
    100% { transform: translate(0); text-shadow: 1px 0 #ff00ff, -1px 0 #00ffff; }
}

@keyframes fadeInOut {
    0% { opacity: 0; }
    10% { opacity: 1; }
    90% { opacity: 1; }
    100% { opacity: 0; }
}

/* =========================
   RESTARTING POPUP
========================= */
#restartingPopup {
    display: none; /* Hidden by default */
    position: fixed;
    left: 50%;
    top: 50%;
    transform: translate(-50%, -50%);
    z-index: 101; /* Higher than warning popups */
    padding: 25px 35px; /* Adjusted padding */
    border: 2px solid #00ffff; /* Cyberpunk blue border, slightly thicker */
    background: #0a0a0a; /* Darker background for contrast */
    color: #00ffcc; /* Cyan/green text for a 'system' feel */
    font-size: 22px; /* Slightly smaller font */
    text-align: center;
    box-shadow: 0 0 30px rgba(0, 255, 255, 0.6);
    min-width: 320px; /* Slightly wider */
    border-radius: 5px; /* Rounded corners for 'window' look */
}

#restartingPopup .popupContent p {
    margin: 8px 0; /* Adjusted margin */
}

.progressBarContainer {
    width: 100%;
    background: #333;
    border: 1px solid #00ffff;
    height: 22px; /* Slightly smaller height */
    margin: 12px 0; /* Adjusted margin */
    overflow: hidden; /* Ensure progress bar stays within bounds */
    border-radius: 3px;
}

#progressBar {
    height: 100%;
    width: 0%;
    background-color: #00ff66; /* Green progress */
    transition: width 0.1s linear; /* Smooth transition */
}

#progressText {
    font-size: 16px; /* Slightly smaller font */
    margin-top: 3px;
    color: #00ff66; /* Match progress bar color */
}

/* =========================
   GLITCH & FREEZE
========================= */

.glitch {
    animation: screenGlitch 0.25s;
}

body.glitch {
    animation: screenGlitch 0.25s;
}

body.freeze {
    overflow: hidden; /* Prevent scrolling */
    pointer-events: none; /* Disable all interactions */
}


@keyframes screenGlitch {
    0% { transform: translate(0); filter: hue-rotate(0deg); }
    10% { transform: translate(-5px, 3px) scale(1.02) skewX(2deg); filter: hue-rotate(30deg) saturate(1.5); }
    20% { transform: translate(3px, -5px) scale(0.98) skewY(3deg); filter: hue-rotate(60deg) contrast(1.2); }
    30% { transform: translate(-4px, 2px) scale(1.01) skewX(-1deg); filter: hue-rotate(90deg) brightness(1.3); }
    40% { transform: translate(2px, -4px) scale(0.99) skewY(-2deg); filter: hue-rotate(120deg) saturate(1.8); }
    50% { transform: translate(-3px, 1px) scale(1.03) skewX(1deg); filter: hue-rotate(150deg) contrast(1.5); }
    60% { transform: translate(1px, -3px) scale(0.97) skewY(2deg); filter: hue-rotate(180deg) brightness(1.5); }
    70% { transform: translate(-2px, 4px) scale(1.005) skewX(-3deg); filter: hue-rotate(210deg) saturate(2.0); }
    80% { transform: translate(4px, -1px) scale(0.995) skewY(-1deg); filter: hue-rotate(240deg) contrast(1.8); }
    90% { transform: translate(-1px, 2px) scale(1.015) skewX(2deg); filter: hue-rotate(270deg) brightness(1.8); }
    100% { transform: translate(0); filter: hue-rotate(360deg); }
}

@keyframes glitchEffect {
    0% { transform: translate(0); text-shadow: 2px 0 #d60000, -2px 0 #00ffff; }
    25% { transform: translate(-2px, 2px); text-shadow: -2px 0 #d60000, 2px 0 #00ffff; }
    50% { transform: translate(2px, -2px); text-shadow: 4px 0 #d60000, -4px 0 #00ffff; }
    75% { transform: translate(-1px, 1px); text-shadow: -1px 0 #d60000, 1px 0 #00ffff; }
    100% { transform: translate(0); text-shadow: 2px 0 #d60000, -2px 0 #00ffff; }
}


/* =========================
   MOBILE
========================= */

@media(max-width:600px) {

    #title {
        font-size: 21px;
    }

    #tagline {
        font-size: 11px;
    }

    #youtubeWindow iframe {
        height: 210px;
    }

    #sideMenu {
        width: 270px;
    }
}

</style>

<link href="https://fonts.googleapis.com/css2?family=Press+Start+2P&display=swap" rel="stylesheet"> <!-- Add glitched font -->

</head>


<body>


<canvas id="matrix"></canvas>


<div id="sideMenu">

    <div class="menuTitleContainer">
        <div class="menuTitle">
            // ALPHA MENU
        </div>
        <button class="menuCloseButton" onclick="toggleMenu()">
            X
        </button>
    </div>

    <button
        class="menuButton"
        onclick="newChat()">
        🆕 New Chat
    </button>

    <button
        id="soloButton"
        class="menuButton active"
        onclick="setMode('solo')">
        💬 Solo Mode
    </button>

    <button
        id="worldButton"
        class="menuButton"
        onclick="setMode('world')">
        🌐 World Mode
    </button>

    <button
        class="menuButton"
        onclick="newWorldChat()">
        🆕 New World Chat
    </button>

    <button
        id="youtubeButton"
        class="menuButton"
        onclick="toggleYouTube()">
        ▶️ YouTube Player: OFF
    </button>

    <!-- Add ID for potential future use -->
    <button
        id="showRecentChatsButton"
        class="menuButton"
        onclick="toggleRecentChatsDisplay()">
        🕘 Recent Chats
    </button>

    <div id="recentChatsContainer" style="
        margin-top: 15px;
        padding-top: 10px;
        border-top: 1px solid #064d24;
        max-height: 200px; /* Example max height */
        overflow-y: auto; /* Scroll if too many chats */
        font-size: 13px;
        color: #00ff66;
        display: none; /* Hidden by default */
    ">
        <!-- Recent chats will be dynamically loaded here -->
        <div style="text-align: center; color: #888;">No recent chats available.</div>
    </div>

    <div style="
        margin-top:25px;
        font-size:12px;
        color:#555;
    ">
        Alpha AI<br>
        SYSTEM ONLINE
    </div>

</div>


<div id="creatorPopup">
    Created by Akshat Mishra
</div>

<div id="restartingPopup">
    <div class="popupContent">
        <p>Page Restarting...</p>
        <div class="progressBarContainer">
            <div id="progressBar"></div>
        </div>
        <p id="progressText">0%</p>
    </div>
</div>

<div id="app">

<div id="hackedOverlay"></div> <!-- New full-screen overlay -->

<header>

    <div
        id="menuButton"
        onclick="toggleMenu()">
        ⋮
    </div>

    <div
        id="title"
        onclick="titleClick()">
        Alpha AI
    </div>

    <div id="tagline">
        it can <span id="capability">answer</span>
    </div>

    <div id="headerWarning">
        ⚠️ Do not click 3 times!
    </div>

</header>


<div id="shadowHack">
    <span class="shadow">SHADOW</span>:<span class="hacked">HACKED</span>
</div>


<div id="main">


<div id="chatArea"></div>


<div id="worldWindow"></div>


<div id="youtubeWindow">

    <div id="videoTitle">
        No video playing
    </div>

    <iframe
        id="youtubeFrame"
        allow="autoplay; encrypted-media"
        allowfullscreen>
    </iframe>

    <div class="playerControls">

        <button onclick="pauseVideo()">
            ⏸ Pause
        </button>

        <button onclick="resumeVideo()">
            ▶ Resume
        </button>

        <button onclick="stopYouTube()">
            ⏹ Stop
        </button>

    </div>

    <div id="queueBox">

        <div id="queueHeader">
            🎵 QUEUE — <span id="queueCount">0</span>
        </div>

        <div id="queueInputRow">

            <input
                id="queueInput"
                placeholder="Add song to queue..."
                autocomplete="off">

            <button
                id="queueAddButton"
                onclick="addQueueFromInput()">
                ADD
            </button>

        </div>

        <div id="queueList">
            <div style="color:#555; padding:5px 0;">
                Queue is empty.
            </div>
        </div>

    </div>

</div>


<div id="inputArea">

    <input
        id="messageInput"
        placeholder="Enter message..."
        autocomplete="off">

    <button
        class="actionButton"
        onclick="startVoice()">
        🎤
    </button>

    <button
        class="actionButton"
        onclick="sendMessage()">
        SEND
    </button>

</div>


</div>

</div>


<script>

/* =========================
   VARIABLES
========================= */

let username =
    localStorage.getItem(
        "alpha_username"
    );

if (!username) {

    username =
        prompt(
            "Enter your Alpha AI username:"
        ) || "Guest";

    localStorage.setItem(
        "alpha_username",
        username
    );
}


let currentMode = "solo";

let youtubeEnabled = false;

let youtubePlayer = null;
let youtubePlayerReady = false;
let pendingVideo = null;
let youtubeQueue = [];
let currentQueueVideo = null;
let queuePlaying = false;

let socket = null;

let titleDanger = false;

let clickCount = 0;
let lastClickTime = 0;
let hackedModeActive = false;
let restartProcessInterval = null;
let restartProgressBarValue = 0;

const RESTART_DURATION_SECONDS = 21; // Total duration for restart progress
const HACKED_TEXT_TIMEOUT = 7000; // 7 seconds before restart popup appears


/* =========================
   ELEMENTS
========================= */

const chatArea =
    document.getElementById(
        "chatArea"
    );

const worldWindow =
    document.getElementById(
        "worldWindow"
    );

const messageInput =
    document.getElementById(
        "messageInput"
    );

const youtubeWindow =
    document.getElementById(
        "youtubeWindow"
    );

const youtubeFrame =
    document.getElementById(
        "youtubeFrame"
    );

const videoTitle =
    document.getElementById(
        "videoTitle"
    );

const queueInput =
    document.getElementById(
        "queueInput"
    );

const queueList =
    document.getElementById(
        "queueList"
    );

const queueCount =
    document.getElementById(
        "queueCount"
    );

const recentChatsContainer = document.getElementById("recentChatsContainer");
const hackedOverlay = document.getElementById('hackedOverlay');


/* =========================
   MENU
========================= */

function toggleMenu() {

    document
        .getElementById("sideMenu")
        .classList.toggle("open");
}


function enableYouTubeForWorld() {

    youtubeEnabled = true;

    const button = document.getElementById("youtubeButton");

    if (button) {
        button.textContent = "▶️ YouTube Player: ON";
        button.classList.add("active");
    }

    youtubeWindow.style.display = "block";
}


/* =========================
   MODE
========================= */

function setMode(mode) {

    currentMode = mode;

    if (mode === "world") {

        chatArea.style.display =
            "none";

        worldWindow.style.display =
            "block";

        document
            .getElementById("worldButton")
            .classList.add("active");

        document
            .getElementById("soloButton")
            .classList.remove("active");

        connectWorld();

    } else {

        worldWindow.style.display =
            "none";

        chatArea.style.display =
            "block";

        document
            .getElementById("soloButton")
            .classList.add("active");

        document
            .getElementById("worldButton")
            .classList.remove("active");
    }

    toggleMenu();
}


/* =========================
   WORLD SOCKET
========================= */

function connectWorld() {

    if (socket) return;

    socket = io();
    console.log("Attempting WebSocket connection...");

    socket.on(
        "connect",
        function() {
            console.log("WebSocket connected!"); // Add log
            socket.emit(
                "join_world",
                {
                    username: username
                }
            );

        }
    );


    socket.on(
        "world_history",
        function(messages) {
            console.log("Received world history:", messages); // Add log
            worldWindow.innerHTML = "";

            messages.forEach(
                addWorldMessage
            );

            scrollWorld();
        }
    );


    socket.on(
        "world_message",
        function(item) {
            console.log("Received world message:", item); // Add log
            addWorldMessage(item);

            scrollWorld();
        }
    );

    // WORLD YOUTUBE: one user's music controls are shared with everyone.
    socket.on("world_player_state", function(state) {
        if (state && state.current) {
            enableYouTubeForWorld();
            playVideoObject(state.current);
        }

        if (state && Array.isArray(state.queue)) {
            youtubeQueue = state.queue.slice();
            renderYouTubeQueue();
        }
    });

    socket.on("world_play", function(video) {
        enableYouTubeForWorld();
        playVideoObject(video);
    });

    socket.on("world_queue_add", function(video) {
        enableYouTubeForWorld();
        youtubeQueue.push(video);
        renderYouTubeQueue();

        if (!queuePlaying && !currentQueueVideo) {
            playNextQueuedSong();
        }
    });

    socket.on("world_queue_replace", function(queue) {
        youtubeQueue = Array.isArray(queue) ? queue.slice() : [];
        renderYouTubeQueue();
    });


    socket.on(
        "system_message",
        function(item) {
            console.log("Received system message:", item); // Add log
            const div =
                document.createElement(
                    "div"
                );

            div.className =
                "message systemMessage";

            div.textContent =
                item.message;

            worldWindow.appendChild(
                div
            );

            scrollWorld();
        }
    );
}


function addWorldMessage(item) {

    const div =
        document.createElement(
            "div"
        );

    div.className =
        "message " +
        (
            item.type === "ai"
            ? "aiMessage" +
            (
                item.username === username
                ? " userMessage"
                : ""
            )
            : ""
        );

    const name =
        document.createElement(
            "strong"
        );

    name.textContent =
        item.username + ": ";

    div.appendChild(name);

    div.appendChild(
        document.createTextNode(
            item.message
        )
    );

    worldWindow.appendChild(div);
}


function scrollWorld() {

    worldWindow.scrollTop =
        worldWindow.scrollHeight;
}


/* =========================
   MESSAGE
========================= */

async function sendMessage() {

    const message =
        messageInput.value.trim();

    if (!message) return;


    /*
       IMPORTANT:
       Detect PLAY commands first.
    */

    const lower =
        message.toLowerCase();


    if (
        lower.startsWith("queue ") ||
        lower.startsWith("queue:")
    ) {

        if (!youtubeEnabled) {

            addMessage(
                "⚠️ Please enable the YouTube Player from the side menu first.",
                "ai"
            );

            messageInput.value = "";

            return;
        }

        const query =
            message
                .replace(
                    /^queue\s*:?\s*/i,
                    ""
                )
                .trim();

        if (!query) {

            addMessage(
                "⚠️ Tell me what you want to add to the queue.",
                "ai"
            );

            messageInput.value = "";

            return;
        }

        await addToQueue(query);

        messageInput.value = "";

        return;
    }


    if (
        lower.startsWith("play ") ||
        lower.startsWith("play:")
    ) {

        if (!youtubeEnabled) {

            addMessage(
                "⚠️ Please enable the YouTube Player from the side menu first.",
                "ai"
            );

            messageInput.value = "";

            return;
        }

        const query =
            message
                .replace(
                    /^play\s*:?\s*/i,
                    ""
                )
                .trim();

        if (!query) {

            addMessage(
                "⚠️ Tell me what you want to play.",
                "ai"
            );

            return;
        }

        await playYouTube(query);

        messageInput.value = "";

        return;
    }


    /* =====================
       WORLD YOUTUBE
    ===================== */

    if (currentMode === "world" && (lower.startsWith("play ") || lower.startsWith("play:") || lower.startsWith("queue ") || lower.startsWith("queue:"))) {

        if (!youtubeEnabled) {
            addMessage(
                "⚠️ Please enable the YouTube Player from the side menu first.",
                "ai"
            );
            messageInput.value = "";
            return;
        }

        const isQueueCommand = lower.startsWith("queue ") || lower.startsWith("queue:");
        const query = message.replace(/^(play|queue)\s*:?[\s]*/i, "").trim();

        if (!query) {
            addMessage("⚠️ Tell me what you want to play.", "ai");
            messageInput.value = "";
            return;
        }

        try {
            videoTitle.textContent = (isQueueCommand ? "Adding to world queue: " : "Searching: ") + query;
            const video = await searchYouTube(query);

            if (isQueueCommand) {
                socket.emit("world_queue_add_request", {video: video});
            } else {
                socket.emit("world_play_request", {video: video});
            }
        } catch (error) {
            addMessage("⚠️ " + error.message, "ai");
            console.error(error);
        }

        messageInput.value = "";
        return;
    }


    /* =====================
       WORLD
    ===================== */

    if (currentMode === "world") {

        if (!socket) {

            connectWorld();

        }

        socket.emit(
            "world_message",
            {
                message: message
            }
        );
        console.log("Sent world message:", message); // Add log
        messageInput.value = "";

        return;
    }


    /* =====================
       SOLO
    ===================== */

    addMessage(
        message,
        "user"
    );

    messageInput.value = "";


    try {

        const response =
            await fetch(
                "/chat",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        username: username,
                        message: message
                    })
                }
            );


        const data =
            await response.json();


        addMessage(
            data.reply,
            "ai"
        );


        saveRecentChat(
            message
        );


    } catch (error) {

        addMessage(
            "⚠️ Connection error. Please try again.",
            "ai"
        );

        console.error(error);
    }
}


/* =========================
   ADD MESSAGE
========================= */

function addMessage(
    text,
    type
) {

    const div =
        document.createElement(
            "div"
        );

    div.className =
        "message " +
        (
            type === "user"
            ? "userMessage"
            : "aiMessage"
        );

    div.textContent =
        text;

    chatArea.appendChild(
        div
    );

    chatArea.scrollTop =
        chatArea.scrollHeight;
}


/* =========================
   NEW CHAT
========================= */

async function newChat() {

    chatArea.innerHTML = "";

    messageInput.value = "";

    stopYouTube();
    clearYouTubeQueue();

    try {

        await fetch(
            "/reset",
            {
                method: "POST",

                headers: {
                    "Content-Type":
                        "application/json"
                },

                body: JSON.stringify({
                    username: username
                })
            }
        );

    } catch (e) {

        console.log(e);
    }

    addMessage(
        "New chat started. 🤖",
        "ai"
    );

    toggleMenu();
}


/* =========================
   NEW WORLD CHAT
========================= */

function newWorldChat() {
    if (currentMode === "world") {
        worldWindow.innerHTML = "";
        // This only clears the client-side display. Server-side world_messages are not reset.
        addWorldMessage({
            username: "System",
            message: "New world chat started. 🌐",
            type: "system"
        });
        scrollWorld();
    } else {
        addMessage(
            "Please switch to World Mode to start a new world chat.",
            "ai"
        );
    }
    toggleMenu(); // Close menu after action
}


/* =========================
   YOUTUBE TOGGLE
========================= */

function toggleYouTube() {

    youtubeEnabled =
        !youtubeEnabled;


    const button =
        document.getElementById(
            "youtubeButton"
        );


    if (youtubeEnabled) {

        button.textContent =
            "▶️ YouTube Player: ON";

        button.classList.add(
            "active"
        );

        youtubeWindow.style.display =
            "block";

    } else {

        button.textContent =
            "▶️ YouTube Player: OFF";

        button.classList.remove(
            "active"
        );

        stopYouTube();
        clearYouTubeQueue();
    }

    toggleMenu();
}


/* =========================
   YOUTUBE QUEUE + PLAYER
========================= */

let youtubeApiReady = false;

function onYouTubeIframeAPIReady() {

    youtubeApiReady = true;
    createYouTubePlayer();
}

function createYouTubePlayer() {

    if (!youtubeApiReady) return;
    if (youtubePlayer) return;

    const frame = document.getElementById("youtubeFrame");
    if (!frame) {
        setTimeout(createYouTubePlayer, 100);
        return;
    }

    youtubePlayer =
        new YT.Player(
            "youtubeFrame",
            {
                playerVars: {
                    autoplay: 0,
                    enablejsapi: 1,
                    rel: 0
                },

                events: {
                    onReady: function() {

                        youtubePlayerReady = true;

                        if (pendingVideo) {
                            const video = pendingVideo;
                            pendingVideo = null;
                            playVideoObject(video);
                        }
                    },

                    onStateChange: function(event) {

                        if (event.data === YT.PlayerState.ENDED) {
                            playNextQueuedSong();
                        }
                    }
                }
            }
        );
}


/* =========================
   SEARCH YOUTUBE
========================= */

async function searchYouTube(query) {

    query = query.trim();

    if (!query) {
        throw new Error("Please enter a song name.");
    }

    const controller = new AbortController();
    const timeout = setTimeout(function() {
        controller.abort();
    }, 8000);

    try {
        const response = await fetch(
            "/music_search",
            {
                method: "POST",
                headers: {
                    "Content-Type": "application/json"
                },
                body: JSON.stringify({ query: query }),
                signal: controller.signal
            }
        );

        if (!response.ok) {
            throw new Error("Music search server error: " + response.status);
        }

        const data = await response.json();

        if (data.error) {
            throw new Error(data.error);
        }

        if (!data.videoId) {
            throw new Error("No YouTube video found.");
        }

        return {
            title: data.title || query,
            videoId: data.videoId
        };

    } catch (error) {
        if (error.name === "AbortError") {
            throw new Error("YouTube search took too long. Check your YouTube API key and try again.");
        }
        throw error;
    } finally {
        clearTimeout(timeout);
    }
}


/* =========================
   PLAY VIDEO OBJECT
========================= */

function playVideoObject(video) {

    if (!youtubeEnabled) {

        return;
    }

    currentQueueVideo =
        video;

    queuePlaying = true;

    youtubeWindow.style.display =
        "block";

    videoTitle.textContent =
        "▶ " + video.title;


    if (
        youtubePlayerReady &&
        youtubePlayer
    ) {

        youtubePlayer.loadVideoById(
            video.videoId
        );

    } else {

        pendingVideo = video;
    }

    renderYouTubeQueue();
}


/* =========================
   PLAY DIRECTLY
========================= */

async function playYouTube(
    query
) {

    if (!youtubeEnabled) {

        addMessage(
            "⚠️ Please enable the YouTube Player from the side menu first.",
            "ai"
        );

        return;
    }


    videoTitle.textContent =
        "Searching: " + query;


    try {

        const video =
            await searchYouTube(
                query
            );

        playVideoObject(video);

    } catch (error) {

        addMessage(
            "⚠️ " + error.message,
            "ai"
        );

        console.error(error);
    }
}


/* =========================
   ADD TO QUEUE
========================= */

async function addToQueue(
    query
) {

    if (!youtubeEnabled) {

        addMessage(
            "⚠️ Please enable the YouTube Player from the side menu first.",
            "ai"
        );

        return;
    }


    try {

        videoTitle.textContent =
            "Adding to queue: " +
            query;


        const video =
            await searchYouTube(
                query
            );


        youtubeQueue.push(
            video
        );


        renderYouTubeQueue();


        addMessage(
            "🎵 Added to queue: " +
            video.title,
            "ai"
        );


        if (!queuePlaying) {

            playNextQueuedSong();
        }

    } catch (error) {

        addMessage(
            "⚠️ " + error.message,
            "ai"
        );

        console.error(error);
    }
}


/* =========================
   QUEUE INPUT
========================= */

async function addQueueFromInput() {

    const query =
        queueInput.value.trim();


    if (!query) {

        return;
    }


    queueInput.value = "";

    if (currentMode === "world") {
        try {
            videoTitle.textContent = "Adding to world queue: " + query;
            const video = await searchYouTube(query);
            socket.emit("world_queue_add_request", {video: video});
        } catch (error) {
            addMessage("⚠️ " + error.message, "ai");
            console.error(error);
        }
        return;
    }

    await addToQueue(query);
}


/* =========================
   PLAY NEXT
========================= */

function playNextQueuedSong() {

    if (!youtubeEnabled) {

        return;
    }


    if (!youtubeQueue.length) {

        queuePlaying = false;

        currentQueueVideo = null;

        videoTitle.textContent =
            "No video playing";

        renderYouTubeQueue();

        return;
    }


    const nextVideo =
        youtubeQueue.shift();


    playVideoObject(
        nextVideo
    );
}


/* =========================
   RENDER QUEUE
========================= */

function renderYouTubeQueue() {

    queueCount.textContent =
        youtubeQueue.length;


    queueList.innerHTML = "";


    if (!youtubeQueue.length) {

        queueList.innerHTML =
            '<div style="color:#555; padding:5px 0;">Queue is empty.</div>';

        return;
    }


    youtubeQueue.forEach(
        function(video, index) {

            const item =
                document.createElement(
                    "div"
                );

            item.className =
                "queueItem";


            const title =
                document.createElement(
                    "div"
                );

            title.className =
                "queueItemTitle";

            title.textContent =
                (index + 1) +
                ". " +
                video.title;


            const removeButton =
                document.createElement(
                    "button"
                );

            removeButton.className =
                "queueRemoveButton";

            removeButton.textContent =
                "X";


            removeButton.onclick =
                function() {

                    if (currentMode === "world" && socket) {
                        socket.emit("world_queue_remove_request", {index: index});
                    } else {
                        youtubeQueue.splice(index, 1);
                        renderYouTubeQueue();
                    }
                };


            item.appendChild(
                title
            );

            item.appendChild(
                removeButton
            );

            queueList.appendChild(
                item
            );
        }
    );
}


/* =========================
   CLEAR QUEUE
========================= */

function clearYouTubeQueue() {

    youtubeQueue = [];

    currentQueueVideo = null;

    queuePlaying = false;

    pendingVideo = null;

    renderYouTubeQueue();
}


/* =========================
   STOP
========================= */

function stopYouTube() {

    if (
        youtubePlayerReady &&
        youtubePlayer
    ) {

        youtubePlayer.stopVideo();

    }


    currentQueueVideo = null;

    queuePlaying = false;

    pendingVideo = null;


    if (youtubeEnabled) {

        youtubeWindow.style.display =
            "block";

    } else {

        youtubeWindow.style.display =
            "none";
    }


    videoTitle.textContent =
        "No video playing";
}


/* =========================
   PAUSE
========================= */

function pauseVideo() {

    if (
        youtubePlayerReady &&
        youtubePlayer
    ) {

        youtubePlayer.pauseVideo();
    }
}


/* =========================
   RESUME
========================= */

function resumeVideo() {

    if (
        youtubePlayerReady &&
        youtubePlayer
    ) {

        youtubePlayer.playVideo();
    }
}


/* =========================
   VOICE
========================= */

function startVoice() {

    const SpeechRecognition =
        window.SpeechRecognition ||
        window.webkitSpeechRecognition;


    if (!SpeechRecognition) {

        addMessage(
            "⚠️ Voice input is not supported by this browser.",
            "ai"
        );

        return;
    }


    const recognition =
        new SpeechRecognition();


    recognition.lang =
        "en-IN";

    recognition.interimResults =
        false;


    recognition.onresult =
        function(event) {

            messageInput.value =
                event.results[0][0].transcript;
        };


    recognition.start();
}


/* =========================
   RECENT CHATS
========================= */

function saveRecentChat(
    message
) {

    let chats =
        JSON.parse(
            localStorage.getItem(
                "alpha_recent_chats"
            ) || "[]"
        );


    chats.unshift({
        text: message,
        time: new Date().toLocaleString()
    });


    chats =
        chats.slice(0,20);


    localStorage.setItem(
        "alpha_recent_chats",
        JSON.stringify(chats)
    );

    // Immediately update the display if menu is open and recent chats are visible
    if (document.getElementById("sideMenu").classList.contains("open") && recentChatsContainer.style.display !== 'none') {
        displayRecentChatsInPanel();
    }
}


function displayRecentChatsInPanel() {
    const chats = JSON.parse(localStorage.getItem("alpha_recent_chats") || "[]");

    recentChatsContainer.innerHTML = ''; // Clear previous entries

    if (!chats.length) {
        recentChatsContainer.innerHTML = '<div style="text-align: center; color: #888;">No recent chats available.</div>';
        return;
    }

    chats.forEach(function(chat, index) {
        const chatItem = document.createElement('div');
        chatItem.className = 'recentChatItem'; // Add a class for styling
        chatItem.style.cssText = `
            padding: 8px 0;
            cursor: pointer;
            border-bottom: 1px dotted #064d24;
            margin-bottom: 5px;
            word-break: break-word;
        `;
        chatItem.innerHTML = `<strong>${chat.time}:</strong> ${chat.text.substring(0, 50)}${chat.text.length > 50 ? '...' : ''}`;
        chatItem.onclick = function() { loadChat(index); };
        recentChatsContainer.appendChild(chatItem);
    });
}

// New function to toggle recent chats display
function toggleRecentChatsDisplay() {
    toggleMenu(); // Ensure the side menu is open
    if (recentChatsContainer.style.display === 'none') {
        displayRecentChatsInPanel(); // Load and display chats
        recentChatsContainer.style.display = 'block'; // Show the container
    } else {
        recentChatsContainer.style.display = 'none'; // Hide the container
    }
}

// Original showRecentChats function is no longer needed, or can be repurposed.
// For now, removing it if toggleRecentChatsDisplay is the new entry point.

// Placeholder for loading a chat - will be implemented in detail later
async function loadChat(index) {
    const chats = JSON.parse(localStorage.getItem("alpha_recent_chats") || "[]");
    if (index >= 0 && index < chats.length) {
        const chatToLoad = chats[index];
        chatArea.innerHTML = ""; // Clear current chat

        // Display the initial message of the loaded chat
        addMessage(`Loading chat session from ${chatToLoad.time}:`, "system");
        addMessage(chatToLoad.text, "user"); // Show the user's initial message
        addMessage("Note: Full historical chat context loading from the backend is not yet implemented. This displays the initial message for reference.", "ai");

        // Do not close the menu here, let the user decide
        // toggleMenu();

        // In a real scenario, you'd fetch/reconstruct the full conversation and update solo_chats.
        // For demonstration, we'll just show the initial message and note.
    }
}


/* =========================
   TITLE
========================= */

function titleClick() {

    titleDanger =
        !titleDanger;


    const title =
        document.getElementById(
            "title"
        );


    if (titleDanger) {

        title.textContent =
            "DANGER";

        title.style.color =
            "#d60000";

    } else {

        title.textContent =
            "Alpha AI";

        title.style.color =
            "#6500b8";
    }


    document.body.classList.add(
        "glitch"
    );


    setTimeout(
        function() {

            document.body.classList.remove(
                "glitch"
            );

        },
        250
    );
}


/* =========================
   TRIPLE CLICK HACKED MODE
========================= */
let warningPopups = []; // Keep track of warning popups

document.body.addEventListener('click', function(event) {
    if (hackedModeActive) return; // Do not trigger if already in hacked mode

    const currentTime = new Date().getTime();

    // Reset click count if too much time has passed between clicks
    if (currentTime - lastClickTime > 500) { // 500ms threshold for triple click
        clickCount = 0;
    }

    clickCount++;
    lastClickTime = currentTime;

    if (clickCount >= 3) {
        clickCount = 0; // Reset
        activateHackedMode();
    }
});

function activateHackedMode() {
    hackedModeActive = true;
    document.body.classList.add('freeze');
    document.body.classList.add('glitch'); // Apply glitch to entire screen

    hackedOverlay.style.display = 'block'; // Show full-screen overlay
    document.getElementById('shadowHack').style.display = 'block'; // Show SHADOW:HACKED
    document.getElementById('shadowHack').classList.add('glitch'); // Apply glitch to shadowHack text

    // Generate multiple warning pop-ups
    for (let i = 0; i < 5; i++) { // Generate 5 pop-ups
        const popup = document.createElement('div');
        popup.className = 'hackedWarningPopup';
        popup.textContent = 'DANGER WARNING: DEVICE HACKED!';
        const x = Math.random() * (window.innerWidth - 300);
        const y = Math.random() * (window.innerHeight - 100);
        popup.style.left = `${x}px`;
        popup.style.top = `${y}px`;
        popup.style.animationDelay = `${i * 0.1}s`; // Stagger animation
        document.body.appendChild(popup);
        warningPopups.push(popup);

        // Make them visible for a short duration with fade effect
        setTimeout(() => {
            popup.style.opacity = '1';
        }, 50 + (i * 100)); // Stagger visibility

        // Then fade out
        setTimeout(() => {
            popup.style.opacity = '0';
            setTimeout(() => popup.remove(), 1000); // Remove after fade out
        }, HACKED_TEXT_TIMEOUT - 1000 + (i * 200)); // Fade out before restart popup
    }

    setTimeout(startRestartingSequence, HACKED_TEXT_TIMEOUT);
}

function startRestartingSequence() {
    hackedOverlay.style.display = 'none'; // Hide overlay
    document.getElementById('shadowHack').style.display = 'none'; // Hide SHADOW:HACKED
    document.getElementById('shadowHack').classList.remove('glitch'); // Remove glitch from shadowHack text

    // Remove any remaining warning popups
    warningPopups.forEach(popup => popup.remove());
    warningPopups = [];

    document.getElementById('restartingPopup').style.display = 'block';

    restartProgressBarValue = 0;
    document.getElementById('progressBar').style.width = '0%';
    document.getElementById('progressText').textContent = '0%';

    let startTime = new Date().getTime();
    const endTime = startTime + (RESTART_DURATION_SECONDS * 1000);

    restartProcessInterval = setInterval(() => {
        const now = new Date().getTime();
        const elapsed = now - startTime;
        let progress = (elapsed / (RESTART_DURATION_SECONDS * 1000)) * 100;

        if (progress >= 100) {
            progress = 100;
            clearInterval(restartProcessInterval);
            finishRestartingSequence();
        }

        document.getElementById('progressBar').style.width = progress.toFixed(0) + '%';
        document.getElementById('progressText').textContent = progress.toFixed(0) + '%';
    }, 100); // Update every 100ms
}

async function finishRestartingSequence() {
    document.body.classList.remove('freeze');
    document.body.classList.remove('glitch'); // Remove glitch from entire screen
    document.getElementById('restartingPopup').style.display = 'none';

    hackedModeActive = false; // Reset hacked mode flag
    await newChat(); // Call newChat to reset the chat, already async
}


/* =========================
   DOUBLE CLICK CREATOR
========================= */

document.addEventListener(
    "dblclick",
    function(event) {

        /*
          Do not trigger from generated
          chat messages.
        */

        if (
            event.target.closest(
                ".message"
            )
        ) {
            return;
        }


        const popup =
            document.getElementById(
                "creatorPopup"
            );


        popup.style.display =
            "block";


        setTimeout(
            function() {

                popup.style.display =
                    "none";

            },
            2500
        );
    }
);


/* =========================
   CAPABILITY ANIMATION
========================= */

const capabilities = [

    "answer",
    "search",
    "play music",
    "play videos",
    "chat",
    "think"

];


let capabilityIndex = 0;


setInterval(
    function() {

        capabilityIndex =
            (
                capabilityIndex + 1
            ) %
            capabilities.length;


        const element =
            document.getElementById(
                "capability"
            );


        element.classList.add(
            "glitch"
        );


        setTimeout(
        function() {

                element.textContent =
                    capabilities[
                        capabilityIndex
                    ];

                element.classList.remove(
                    "glitch"
                );

            },
            150
        );

    },
    1800
);


/* =========================
   ENTER KEY
========================= */

messageInput.addEventListener(
    "keydown",
    function(event) {

        if (
            event.key === "Enter"
        ) {

            sendMessage();
        }
    }
);


/* =========================
   MATRIX RAIN
========================= */

const canvas =
    document.getElementById(
        "matrix"
    );

const ctx =
    canvas.getContext(
        "2d"
    );


function resizeMatrix() {

    canvas.width =
        window.innerWidth;

    canvas.height =
        window.innerHeight;
}


resizeMatrix();

window.addEventListener(
    "resize",
    resizeMatrix
);


const chars =
    "01ABCDEFGHIJKLMNOPQRSTUVWXYZ#$%";


let fontSize = 14;

let columns =
    Math.floor(
        window.innerWidth /
        fontSize
    );


let drops =
    Array(columns).fill(1);


function matrixRain() {

    ctx.fillStyle =
        "rgba(0,0,0,0.06)";

    ctx.fillRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    ctx.font =
        fontSize + "px monospace";


    for (
        let i = 0;
        i < drops.length;
        i++
    ) {

        const text =
            chars[
                Math.floor(
                    Math.random() *
                    chars.length
                )
            ];


        ctx.fillStyle =
            "#00ff66";


        ctx.fillText(
            text,
            i * fontSize,
            drops[i] * fontSize
        );


        if (
            drops[i] * fontSize >
            canvas.height &&
            Math.random() > 0.975
        ) {

            drops[i] = 0;
        }


        drops[i]++;
    }
}


setInterval(
    matrixRain,
    50
);


/* =========================
   START MESSAGE
========================= */

addMessage(
    "Alpha AI online. 🤖\nType a message to begin.",
    "ai"
);

// Initial call to populate recent chats if any exist when the page loads
document.addEventListener('DOMContentLoaded', function() {
    displayRecentChatsInPanel();
    createYouTubePlayer();
});

// Initialize the empty YouTube queue
renderYouTubeQueue();

// Enter inside the queue box adds the song
queueInput.addEventListener(
    "keydown",
    function(event) {

        if (event.key === "Enter") {

            addQueueFromInput();
        }
    }
);

</script>

</body>

</html>
'''


with open(
    "index.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(html_code)


print("✅ index.html created!")

In [ ]:
import threading
import time
import requests
import os
import subprocess

print("Executing server startup script...")

# Function to run the Flask server
def run_server(app_instance, socketio_instance):
    print("Starting Flask-SocketIO server...")
    socketio_instance.run(
        app_instance,
        host="0.0.0.0",
        port=5000,
        allow_unsafe_werkzeug=True
    )
    print("Flask-SocketIO server stopped.")

# Removed the lsof process killing block for now to ensure output visibility.
# If you encounter 'Address already in use' errors, you may need to restart the Colab runtime manually.
print("Skipping port 5000 cleanup for now. If you get 'Address already in use', restart Colab runtime.")

# Give a small delay before starting the server
time.sleep(2)

# Start a new server thread, passing app and socketio as arguments
server_thread = threading.Thread(
    target=run_server,
    args=(app, socketio), # Pass app and socketio here
    daemon=True # Daemon threads exit when the main program exits
)

server_thread.start()

time.sleep(10) # Give the server more time to start and bind to the port

# Test the Flask server to ensure it's running
try:
    response = requests.get(
        "http://127.0.0.1:5000/",
        timeout=10
    )
    print(
        "✅ Flask server is working!"
    )
    print(
        "Status:",
        response.status_code
    )
except Exception as e:
    print(
        "❌ Server test failed:",
        e
    )
    print("It's possible the server failed to start or the port is still in use.")
    print("Please check the Colab runtime logs for Flask/SocketIO startup errors.")
    print("You might need to restart the Colab runtime (Runtime -> Restart runtime...) if the issue persists.")

In [ ]:
import subprocess
import time
import re
import os

print("🌐 Starting Cloudflare Tunnel...")

os.system(
    "pkill -9 cloudflared 2>/dev/null"
)

time.sleep(2)


if not os.path.exists(
    "cloudflared"
):

    os.system(
        "wget -q "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 "
        "-O cloudflared"
    )

    os.chmod(
        "cloudflared",
        0o755
    )


cloudflare_process = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:5000"
    ],

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True
)


public_url = None


while True:

    line = (
        cloudflare_process
        .stdout
        .readline()
    )

    if not line:
        continue


    print(
        line.strip()
    )


    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )


    if match:
        public_url = match.group(0)

        print(
            "\n================================"
        )

        print(
            "🚀 ALPHA AI PUBLIC URL:"
        )

        print(
            public_url
        )

        print(
            "================================"
        )

        break